# SDXL 建筑材质增强基线

输入为设计师已有的基础材质渲染图，SDXL 只做低强度 Img2Img，比较 `0.15 / 0.25 / 0.35` 三档重绘强度。模型缓存和生成结果都写入 Google Drive。

## 1. 挂载 Drive 并设置持久化缓存

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ifc-ai-render')
CACHE_DIR = Path('/content/drive/MyDrive/ifc-ai-render-cache/huggingface')
RESULT_DIR = DRIVE_ROOT / 'results' / 'sdxl_rgb_baseline'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(CACHE_DIR / 'hub')
print('模型缓存:', CACHE_DIR)
print('实验结果:', RESULT_DIR)

## 2. 确认 GPU 并获取项目

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 Colab 的运行时设置中启用 GPU'
print('当前 GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

REPO_URL = 'https://github.com/StrawberryChen/ifc-ai-render.git'
REPO_DIR = Path('/content/ifc-ai-render')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('当前代码版本:')
!git log -1 --oneline

## 3. 安装依赖（安装后若 Colab 提示重启，重启后从第1节重新运行）

In [ ]:
!pip -q install -r requirements-colab.txt

## 4. 查看并调整实验配置

正常情况下只修改下面的 `overrides`。完整默认值保存在 `configs/sdxl_img2img_baseline.json`。

In [ ]:
import json

CONFIG_PATH = REPO_DIR / 'configs' / 'sdxl_img2img_baseline.json'
RUN_CONFIG_PATH = Path('/content/sdxl_img2img_run.json')
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

# 日常实验主要修改这里
overrides = {
    'input_image': str(REPO_DIR / 'outputs' / 'sdcc' / 'sdcc.material.png'),
    'output_directory': str(RESULT_DIR),
    'seed': 42,
    'strengths': [0.15, 0.25, 0.35],
    'steps': 50,
    'guidance_scale': 5.0,
}
config['input']['image'] = overrides['input_image']
config['output']['directory'] = overrides['output_directory']
config['inference']['seed'] = overrides['seed']
config['inference']['strengths'] = overrides['strengths']
config['inference']['num_inference_steps'] = overrides['steps']
config['inference']['guidance_scale'] = overrides['guidance_scale']
RUN_CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(RUN_CONFIG_PATH.read_text(encoding='utf-8'))

## 5. 先校验路径和配置，不下载模型

In [ ]:
!python inference/generate_sdxl_img2img.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR" \
  --validate-only

## 6. 运行三档基线

第一次执行会将 SDXL 权重下载到 Drive。A100 40GB 无需开启 CPU offload。

In [ ]:
!python inference/generate_sdxl_img2img.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR"

## 7. 并排查看输入与结果

In [ ]:
from PIL import Image
from IPython.display import display

display(Image.open(config['input']['image']))
for result_path in sorted(RESULT_DIR.glob('sdcc.strength_*.png')):
    print(result_path.name)
    display(Image.open(result_path))
print((RESULT_DIR / 'run_metadata.json').read_text(encoding='utf-8'))

## 8. 检查 Drive 中的模型和结果占用

In [ ]:
!du -sh "$CACHE_DIR" "$RESULT_DIR" 2>/dev/null || true
!find "$RESULT_DIR" -maxdepth 1 -type f -print